# 4. Pre-Training on Unlabeled Data

Now, we're about to train the GPT arquitecture developed before to predict text consistently.

We're saying that the corpus is unlabeled, but in reality we're using:

input:  [A, B, C, D]

target: [B, C, D, E]

## Pretraining roadmap

This notebook will cover the training phase of the GPT model built in the previous notebooks:

1. **Calculate the next-token loss** for one batch using the model logits and cross-entropy.
2. **Calculate the average loss** across a data loader so training and validation performance can be compared.
3. **Measure the initial losses** before updating the model to establish a baseline.
4. **Create the optimizer** and configure the learning rate and other training hyperparameters.
5. **Run the training loop**: clear gradients, perform the forward pass, calculate the loss, backpropagate, and update the weights for every batch.
6. **Evaluate at regular intervals** on both the training and validation sets with gradient tracking disabled.
7. **Generate sample text during training** from a fixed prompt to inspect the model's progress.
8. **Save the trained model weights** so the model can be loaded and used later.

### 1. Next-token prediction loss

Pretraining uses self-supervised learning. The text does not contain manually
written labels: each token following an input position becomes its target.

The model produces one logit for every vocabulary token at every position.
Cross-entropy measures how much probability the model assigns to the correct
next token.

In [16]:
from pathlib import Path
import sys

import tiktoken
import torch
import torch.nn.functional as F

In [17]:
Path.cwd()

PosixPath('/Users/sergiogomez/Repos/llm-lab/01_pretraining/mini_gpt/notebooks')

In [18]:
src_path = (Path.cwd().parent / "src").resolve()

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

src_path

PosixPath('/Users/sergiogomez/Repos/llm-lab/01_pretraining/mini_gpt/src')

In [19]:
from mini_gpt import GPT_CONFIG_124M, GPTModel

GPT_CONFIG_124M

{'vocab_size': 50257,
 'context_length': 1024,
 'emb_dim': 768,
 'n_heads': 12,
 'n_layers': 12,
 'drop_rate': 0.1,
 'qkv_bias': False}

Each target is the corresponding input shifted by one token. With `B=2` and `T=3`, this batch contains six next-token predictions.

In [20]:
tokenizer = tiktoken.get_encoding("gpt2")

input_batch = torch.tensor([
    [16833, 3626, 6100],  # "every effort moves"
    [40, 1107, 588],      # "I really like"
])

target_batch = torch.tensor([
    [3626, 6100, 345],    # " effort moves you"
    [1107, 588, 11311],   # " really like chocolate"
])

print("Input shape:", input_batch.shape)
print("Target shape:", target_batch.shape)

for input_ids, target_ids in zip(input_batch, target_batch):
    print(
        tokenizer.decode(input_ids.tolist()),
        "->",
        tokenizer.decode(target_ids.tolist()),
    )

Input shape: torch.Size([2, 3])
Target shape: torch.Size([2, 3])
every effort moves ->  effort moves you
I really like ->  really like chocolate


In [21]:
torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (token_embedding): Embedding(50257, 768)
  (positional_embedding): Embedding(1024, 768)
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (attention): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (feed_forward): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attention): Mu

The model returns one logit for every vocabulary token at every position: `(B, T, V)`. Softmax converts those scores into probabilities for inspection.

In [22]:
with torch.no_grad():
    logits = model(input_batch)

probas = torch.softmax(logits, dim=-1)

print("Logits shape:", logits.shape)
print("Probabilities shape:", probas.shape)

Logits shape: torch.Size([2, 3, 50257])
Probabilities shape: torch.Size([2, 3, 50257])


In [23]:
predicted_token_ids = torch.argmax(
    probas,
    dim=-1,
)

print("Predicted token IDs:")
print(predicted_token_ids)

Predicted token IDs:
tensor([[36397, 39619, 20610],
        [ 8615, 49289, 47105]])


In [24]:
for predicted_ids, target_ids in zip(
    predicted_token_ids,
    target_batch,
):
    print("Predicted:", tokenizer.decode(predicted_ids.tolist()))
    print("Target:   ", tokenizer.decode(target_ids.tolist()))
    print()

Predicted:  Gathering SerbianFriday
Target:     effort moves you

Predicted:  cos slicing Aux
Target:     really like chocolate



The random model's most likely tokens are not expected to match the targets. For the loss, we need the probability assigned to each correct target, not only the `argmax` prediction.

In [25]:
target_probas_1 = probas[
    0,
    [0, 1, 2],
    target_batch[0],
]

target_probas_2 = probas[
    1,
    [0, 1, 2],
    target_batch[1],
]

print("Target probabilities, sequence 1:", target_probas_1)
print("Target probabilities, sequence 2:", target_probas_2)

Target probabilities, sequence 1: tensor([2.3466e-05, 2.0531e-05, 1.1733e-05])
Target probabilities, sequence 2: tensor([4.2794e-05, 1.6248e-05, 1.1586e-05])


The mean negative log-likelihood is `-mean(log(p_correct))`: assigning more probability to the correct tokens lowers the loss. With one correct token per position, this is the cross-entropy loss. Perplexity is `exp(loss)`.

In [26]:
target_probas = torch.cat(
    (target_probas_1, target_probas_2)
)

log_probas = torch.log(target_probas)

average_log_proba = torch.mean(log_probas)
negative_average_log_proba = -average_log_proba

print("Target probabilities:", target_probas)
print("Log probabilities:", log_probas)
print("Average log probability:", average_log_proba)
print("Negative average log probability:", negative_average_log_proba)

Target probabilities: tensor([2.3466e-05, 2.0531e-05, 1.1733e-05, 4.2794e-05, 1.6248e-05, 1.1586e-05])
Log probabilities: tensor([-10.6600, -10.7936, -11.3531, -10.0591, -11.0276, -11.3658])
Average log probability: tensor(-10.8765)
Negative average log probability: tensor(10.8765)


### 2. Using PyTorch to compute the loss

Now, we're about calculating the loss using the cross-entropy

In [27]:
logits_flat = logits.flatten(0, 1)
targets_flat = target_batch.flatten()

print("Original logits:", logits.shape)
print("Flattened logits:", logits_flat.shape)

print("Original targets:", target_batch.shape)
print("Flattened targets:", targets_flat.shape)

Original logits: torch.Size([2, 3, 50257])
Flattened logits: torch.Size([6, 50257])
Original targets: torch.Size([2, 3])
Flattened targets: torch.Size([6])


In [28]:
loss = F.cross_entropy(
    logits_flat,
    targets_flat,
)

perplexity = torch.exp(loss)

print("Manual loss:", negative_average_log_proba)
print("PyTorch cross-entropy:", loss)
print("Perplexity:", perplexity)

Manual loss: tensor(10.8765)
PyTorch cross-entropy: tensor(10.8765)
Perplexity: tensor(52918.7734)
